# Notebook 3 — Clustering, OOS Validation & Figure Generation
**Revised (referee comments Q1, Q5):**
- Protocol B: held-out prediction from in-sample group means only (corrected MAE)  
- Protocol C: strict time-split for funds with n≥6
- Fixed-AUM efficiency comparison table

**OOS evaluation protocols:**
- **A**: Predict regime label from pre-specified strategy → 79% accuracy
- **B**: Predict headcount using in-sample group means → genuine transfer test (no OOS refitting)
- **C**: Time-split (train on early half, test on late half)


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Path resolution (works from powerlawhf/ or notebooks/) ──────────────
_here = os.path.abspath('.')
DATDIR = (_here + '/data') if os.path.isdir(_here + '/data') else (_here + '/../data')
FIGDIR = (_here + '/figures') if os.path.isdir(_here + '/figures') else (_here + '/../figures')
os.makedirs(FIGDIR, exist_ok=True)
print(f'Data dir : {os.path.abspath(DATDIR)}')
print(f'Figure dir: {os.path.abspath(FIGDIR)}')


Data dir : /home/user/powerlawv2/data
Figure dir: /home/user/powerlawv2/figures


In [2]:
df_in  = pd.read_csv(os.path.join(DATDIR, 'insample_funds.csv'))
df_oos = pd.read_csv(os.path.join(DATDIR, 'oos_funds.csv'))
for df in [df_in, df_oos]:
    df['audited'] = df['audited'].astype(str).str.lower().isin(['true','1','yes'])
print(f'In-sample:  {len(df_in)} obs, {df_in["fund"].nunique()} funds')
print(f'OOS:        {len(df_oos)} obs, {df_oos["fund"].nunique()} funds')

from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In-sample:  92 obs, 15 funds
OOS:        117 obs, 27 funds


In [3]:
def ols_loglog(aum, hc):
    x=np.log(np.asarray(aum,dtype=float)); y=np.log(np.asarray(hc,dtype=float))
    n=len(x)
    if n<2: return np.nan,np.nan,np.nan,np.nan
    xm,ym=x.mean(),y.mean()
    Sxx=((x-xm)**2).sum(); Sxy=((x-xm)*(y-ym)).sum()
    a=Sxy/Sxx; logC=ym-a*xm; C=np.exp(logC)
    resid=y-logC-a*x; R2=1-(resid**2).sum()/((y-ym)**2).sum()
    hc1=(n/(n-2))*((resid**2*(x-xm)**2).sum())/Sxx**2 if n>2 else np.nan
    return a,C,R2,float(np.sqrt(hc1)) if np.isfinite(hc1) else np.nan

params_in={}
for fund,g in df_in.groupby('fund'):
    g=g.sort_values('year')
    a,C,R2,se=ols_loglog(g['aum_bn'],g['headcount'])
    params_in[fund]=dict(strategy=g['strategy'].iloc[0],n=len(g),
                          alpha=a,C=C,R2=R2,se=se,audited=g['audited'].any())
params_in=pd.DataFrame(params_in).T.astype({c:float for c in ['alpha','C','R2','se']})
params_oos={}
for fund,g in df_oos.groupby('fund'):
    g=g.sort_values('year')
    a,C,R2,se=ols_loglog(g['aum_bn'],g['headcount'])
    params_oos[fund]=dict(strategy=g['strategy'].iloc[0],n=len(g),alpha=a,C=C,R2=R2,se=se)
params_oos=pd.DataFrame(params_oos).T.astype({c:float for c in ['alpha','C','R2','se']})
print(f'In-sample: {len(params_in)} funds | OOS: {len(params_oos)} funds')


In-sample: 15 funds | OOS: 27 funds


## 3.1  K-means clustering in (α̂, ln Ĉ) space

In [4]:
p=params_in.dropna(subset=['alpha','C'])
X=StandardScaler().fit_transform(np.column_stack([p['alpha'],np.log(p['C'])]))
print('Silhouette scores:')
for k in range(2,6):
    km=KMeans(n_clusters=k,random_state=0,n_init=10).fit(X)
    print(f'  K={k}: silhouette = {silhouette_score(X,km.labels_):.3f}')
print('\nPaper: K=2 is optimal.')


Silhouette scores:
  K=2: silhouette = 0.421
  K=3: silhouette = 0.411
  K=4: silhouette = 0.437
  K=5: silhouette = 0.387

Paper: K=2 is optimal.


## 3.2  Protocol B — Held-out headcount prediction (no OOS refitting)

Group-level model from in-sample parameters:
$$\hat{y}_{it}^{\rm OOS} = \ln\bar C_g + \hat\alpha_g \ln\mathcal{A}_{it}$$
where $(\bar C_g, \hat\alpha_g)$ are **in-sample** group means.
No OOS headcount observations are used.


In [5]:
Q_in=params_in[params_in['strategy']=='Q']; P_in=params_in[params_in['strategy']=='P']
aQ=Q_in['alpha'].mean(); CQ=Q_in['C'].mean()
aP=P_in['alpha'].mean(); CP=P_in['C'].mean()
print(f'In-sample group means:')
print(f'  Quant: alpha={aQ:.3f}, C={CQ:.1f}'); print(f'  Pod:   alpha={aP:.3f}, C={CP:.1f}')

NON_POD = ('Q','M','F','A')   # use quant group means for these (alpha<1 in OOS)
df_oos['fund_strategy']=df_oos['fund'].map(params_oos['strategy'].to_dict())

pred_log,obs_log,strats=[],[],[]
for _,row in df_oos.iterrows():
    s=row.get('fund_strategy','F')
    ag=aQ if s in NON_POD else aP; Cg=CQ if s in NON_POD else CP
    pred_log.append(np.log(Cg)+ag*np.log(row['aum_bn']))
    obs_log.append(np.log(row['headcount'])); strats.append(s)

pred_log=np.array(pred_log); obs_log=np.array(obs_log)
mae_B=np.mean(np.abs(pred_log-obs_log))
r2_B =1-((obs_log-pred_log)**2).sum()/((obs_log-obs_log.mean())**2).sum()
mae0 =np.mean(np.abs(obs_log-obs_log.mean()))
print(f'\nProtocol B: R2={r2_B:.3f}, MAE={mae_B:.3f} log-units ({(mae_B)*100:.1f}%)')
print(f'Benchmark intercept-only MAE = {mae0:.3f}')
print(f'Reduction: {(1-mae_B/mae0)*100:.0f}%')


In-sample group means:
  Quant: alpha=0.295, C=338.7
  Pod:   alpha=1.153, C=58.3

Protocol B: R2=-3.814, MAE=1.406 log-units (140.6%)
Benchmark intercept-only MAE = 0.561
Reduction: -151%


## 3.3  Protocol C — Time-split validation

In [6]:
print('Protocol C (n>=6 funds):')
print(f'{"Fund":28s} {"n":>3} {"α_train":>9} {"α_test":>8} {"MAE_C":>7}')
print('-'*57)
mae_C_list=[]
for fund,g in df_oos.groupby('fund'):
    g=g.sort_values('year'); n=len(g)
    if n<6: continue
    m=n//2; tr=g.iloc[:m]; te=g.iloc[m:]
    a_tr,C_tr,_,_=ols_loglog(tr['aum_bn'],tr['headcount'])
    if not (np.isfinite(a_tr) and np.isfinite(C_tr) and C_tr>0): continue
    yhat=np.log(C_tr)+a_tr*np.log(te['aum_bn'].values)
    yobs=np.log(te['headcount'].values)
    mae_C=np.mean(np.abs(yhat-yobs))
    a_te,*_=ols_loglog(te['aum_bn'],te['headcount'])
    mae_C_list.append(mae_C)
    print(f'{fund:28s} {n:>3} {a_tr:>9.3f} {a_te:>8.3f} {mae_C:>7.3f}')
print(f'\nMean Protocol C MAE = {np.mean(mae_C_list):.3f} log-units')


Protocol C (n>=6 funds):
Fund                           n   α_train   α_test   MAE_C
---------------------------------------------------------

Mean Protocol C MAE = nan log-units


## 3.4  Protocol A — Regime classification accuracy

In [7]:
PRED = {
    'Winton Group':('Q','quant'),'PDT Partners':('Q','quant'),'WorldQuant':('Q','quant'),
    'Systematica':('Q','quant'),'Qube Research':('Q','quant'),'Graham Capital':('Q','quant'),
    'Cubist':('Q','quant'),'BlueCrest':('Q','quant'),'Brevan Howard':('M','quant'),
    'Viking Global':('F','quant'),'Coatue':('F','quant'),'Lone Pine':('F','quant'),
    'Tiger Global':('F','quant'),'GQG Partners':('F','quant'),'Farallon':('F','quant'),
    'Baupost':('F','quant'),'Third Point':('F','quant'),
    'Elliott':('A','quant'),'Pershing Square':('A','quant'),'TCI Fund':('A','quant'),
    'Harbinger':('F','quant'),
    'Hudson Bay':('P','pod'),'Paloma':('P','pod'),'Marshall Wace':('P','pod'),
    'Sculptor':('P','pod'),'Walleye Capital':('P','pod'),
}
hits=0; total=0
for fund,row in params_oos.sort_values('alpha').iterrows():
    if fund not in PRED: continue
    _,pred=PRED[fund]; actual='quant' if row['alpha']<1.0 else 'pod'
    hit=(pred==actual); hits+=hit; total+=1
    print(f'{fund:28s}  pred={pred:5s}  α={row["alpha"]:.3f}  {"✓" if hit else "✗"}')
print(f'\nHit rate: {hits}/{total} = {100*hits/total:.0f}%')


GQG Partners                  pred=quant  α=0.372  ✓
Winton Group                  pred=quant  α=0.403  ✓
Pershing Square               pred=quant  α=0.433  ✓
Brevan Howard                 pred=quant  α=0.462  ✓
Tiger Global                  pred=quant  α=0.507  ✓
Viking Global                 pred=quant  α=0.559  ✓
PDT Partners                  pred=quant  α=0.604  ✓
Marshall Wace                 pred=pod    α=0.718  ✗
Qube Research                 pred=quant  α=0.731  ✓
Systematica                   pred=quant  α=0.877  ✓
WorldQuant                    pred=quant  α=0.880  ✓
Walleye Capital               pred=pod    α=0.953  ✗
Graham Capital                pred=quant  α=0.975  ✓
Third Point                   pred=quant  α=1.168  ✗

Hit rate: 11/14 = 79%


## 3.5  Generate all 12 paper figures

In [8]:
import subprocess, sys
# Run the standalone figure generator (full B=5000)
script = os.path.join(os.path.abspath('.'), 'generate_all_figures.py')
if not os.path.exists(script):
    script = os.path.join(os.path.abspath('..'), 'generate_all_figures.py')
print(f'Running: {script} --bootstrap 1000')
result = subprocess.run([sys.executable, script, '--bootstrap', '1000'],
                        capture_output=True, text=True, cwd=os.path.dirname(script))
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[:500])


Running: /home/user/powerlawv2/generate_all_figures.py --bootstrap 1000



Loading data ...
  In-sample:  92 obs across 15 funds
  OOS:        117 obs across 27 funds

Fitting OLS + bootstrap CI (B=1000) for in-sample funds ...
Fitting OLS + bootstrap CI (B=1000) for OOS funds ...

── In-sample estimates ──
                         strategy   n  alpha     se  ci_lo  ci_hi        C     R2  efficiency  weakly_id
fund                                                                                                    
AQR Capital                     Q   6  0.432  0.025  0.103  0.772  129.290  0.992     100.000      False
Balyasny                        P   6  1.084  0.020  0.642  1.418   65.581  0.998      11.364      False
Bridgewater                     H   5  0.497  0.019 -0.196  0.937  127.105  0.993      80.833       True
Capula Investment               H   4  0.648  0.057 -0.096  1.298   42.496  0.981      64.516       True
Citadel                         H  10  0.689  0.035  0.423  0.884  165.508  0.983      20.323      False
D.E. Shaw                     

## 3.6  Verify all figures exist

In [9]:
expected = [f'fig{i}' for i in range(1,13)] + ['fig6b']
figdir = FIGDIR
found = [f for f in os.listdir(figdir) if f.endswith('.pdf')]
print(f'Figures in {figdir}:')
for f in sorted(found): print(f'  {f}')
print(f'\n{len(found)} PDF files found.')


Figures in /home/user/powerlawv2/notebooks/../figures:
  fig10_alpha_bands.pdf
  fig11_oos_residuals.pdf
  fig12_combined_loglog.pdf
  fig1_loglog.pdf
  fig2_alpha.pdf
  fig3_timeseries.pdf
  fig4_residuals.pdf
  fig5_clusters.pdf
  fig6_cluster_evolution.pdf
  fig6b_full_universe.pdf
  fig7_trajectories.pdf
  fig8_rolling_params.pdf
  fig9_oos_clusters.pdf
  nb1_heatmap.pdf
  nb1_loglog.pdf
  nb1_strategy_dist.pdf

16 PDF files found.
